# SQL Exploratory Data Analysis
## Telco Customer Churn Dataset

This notebook runs production-quality SQL queries against the customers table to extract business insights.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from IPython.display import display, Markdown

# Connect to database
engine = create_engine('sqlite:///../data/processed/telco.db')
print("✓ Connected to database")

---
## Query 1: Overall Churn Rate
**BUSINESS INSIGHT:** Overall churn rate as a percentage

In [ ]:
query1 = """
SELECT 
    ROUND(AVG(Churn) * 100, 2) as churn_rate_percent,
    SUM(Churn) as churned_customers,
    COUNT(*) as total_customers
FROM customers;
"""
df1 = pd.read_sql(query1, engine)
display(df1)

---
## Query 2: Churn Rate by Contract Type
**BUSINESS INSIGHT:** Churn rate by Contract type (Month-to-month vs One year vs Two year)

In [ ]:
query2 = """
SELECT 
    Contract,
    COUNT(*) as customer_count,
    SUM(Churn) as churned_count,
    ROUND(AVG(Churn) * 100, 2) as churn_rate_percent
FROM customers
GROUP BY Contract
ORDER BY churn_rate_percent DESC;
"""
df2 = pd.read_sql(query2, engine)
display(df2)

---
## Query 3: Average Monthly Charges
**BUSINESS INSIGHT:** Average MonthlyCharges for churned vs retained customers

In [ ]:
query3 = """
SELECT 
    CASE WHEN Churn = 1 THEN 'Churned' ELSE 'Retained' END as customer_status,
    COUNT(*) as customer_count,
    ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges,
    ROUND(MIN(MonthlyCharges), 2) as min_monthly_charges,
    ROUND(MAX(MonthlyCharges), 2) as max_monthly_charges
FROM customers
GROUP BY Churn
ORDER BY Churn DESC;
"""
df3 = pd.read_sql(query3, engine)
display(df3)

---
## Query 4: Churn Rate by Tenure Bucket
**BUSINESS INSIGHT:** Churn rate by tenure bucket (0-12, 13-24, 25-48, 49+ months)

In [ ]:
query4 = """
SELECT 
    CASE 
        WHEN tenure <= 12 THEN '0-12 months'
        WHEN tenure <= 24 THEN '13-24 months'
        WHEN tenure <= 48 THEN '25-48 months'
        ELSE '49+ months'
    END as tenure_bucket,
    COUNT(*) as customer_count,
    SUM(Churn) as churned_count,
    ROUND(AVG(Churn) * 100, 2) as churn_rate_percent
FROM customers
GROUP BY tenure_bucket
ORDER BY 
    CASE 
        WHEN tenure <= 12 THEN 1
        WHEN tenure <= 24 THEN 2
        WHEN tenure <= 48 THEN 3
        ELSE 4
    END;
"""
df4 = pd.read_sql(query4, engine)
display(df4)

---
## Query 5: Top Service Combinations
**BUSINESS INSIGHT:** Top 5 service combinations among churned customers (InternetService + Contract)

In [ ]:
query5 = """
SELECT 
    InternetService,
    Contract,
    COUNT(*) as churned_customer_count,
    ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges
FROM customers
WHERE Churn = 1
GROUP BY InternetService, Contract
ORDER BY churned_customer_count DESC
LIMIT 5;
"""
df5 = pd.read_sql(query5, engine)
display(df5)

---
## Query 6: Churn Rate by Payment Method
**BUSINESS INSIGHT:** Churn rate by PaymentMethod

In [ ]:
query6 = """
SELECT 
    PaymentMethod,
    COUNT(*) as customer_count,
    SUM(Churn) as churned_count,
    ROUND(AVG(Churn) * 100, 2) as churn_rate_percent
FROM customers
GROUP BY PaymentMethod
ORDER BY churn_rate_percent DESC;
"""
df6 = pd.read_sql(query6, engine)
display(df6)

---
## Query 7: Revenue at Risk
**BUSINESS INSIGHT:** Total annualized MonthlyCharges for all churned customers

In [ ]:
query7 = """
SELECT 
    SUM(Churn) as churned_customers,
    ROUND(SUM(CASE WHEN Churn = 1 THEN MonthlyCharges ELSE 0 END), 2) as monthly_revenue_at_risk,
    ROUND(SUM(CASE WHEN Churn = 1 THEN MonthlyCharges * 12 ELSE 0 END), 2) as annualized_revenue_at_risk
FROM customers;
"""
df7 = pd.read_sql(query7, engine)
display(df7)

---
## Query 8: Customer Risk Segments
**BUSINESS INSIGHT:** HIGH_RISK (tenure<12 AND MonthlyCharges>65), MEDIUM_RISK (tenure<24), LOW_RISK

In [ ]:
query8 = """
SELECT 
    CASE 
        WHEN tenure < 12 AND MonthlyCharges > 65 THEN 'HIGH_RISK'
        WHEN tenure < 24 THEN 'MEDIUM_RISK'
        ELSE 'LOW_RISK'
    END as risk_segment,
    COUNT(*) as customer_count,
    SUM(Churn) as churned_count,
    ROUND(AVG(Churn) * 100, 2) as churn_rate_percent,
    ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges
FROM customers
GROUP BY risk_segment
ORDER BY churn_rate_percent DESC;
"""
df8 = pd.read_sql(query8, engine)
display(df8)

---
## Query 9: Support Services Impact
**BUSINESS INSIGHT:** Churn rate for customers with NO OnlineSecurity AND NO TechSupport

In [ ]:
query9 = """
SELECT 
    CASE 
        WHEN OnlineSecurity = 'No' AND TechSupport = 'No' THEN 'No Security & No Support'
        ELSE 'Has Security or Support'
    END as support_status,
    COUNT(*) as customer_count,
    SUM(Churn) as churned_count,
    ROUND(AVG(Churn) * 100, 2) as churn_rate_percent
FROM customers
GROUP BY support_status
ORDER BY churn_rate_percent DESC;
"""
df9 = pd.read_sql(query9, engine)
display(df9)

---
## Query 10: Service Adoption Analysis
**BUSINESS INSIGHT:** Average number of services subscribed for churned vs retained customers

In [ ]:
query10 = """
SELECT 
    CASE WHEN Churn = 1 THEN 'Churned' ELSE 'Retained' END as customer_status,
    COUNT(*) as customer_count,
    ROUND(AVG(
        CASE WHEN PhoneService = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN MultipleLines = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN OnlineSecurity = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN OnlineBackup = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN DeviceProtection = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN TechSupport = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN StreamingTV = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN StreamingMovies = 'Yes' THEN 1 ELSE 0 END
    ), 2) as avg_services_subscribed
FROM customers
GROUP BY Churn
ORDER BY Churn DESC;
"""
df10 = pd.read_sql(query10, engine)
display(df10)

---
## Summary

All 10 production-quality SQL queries have been executed successfully. Key insights include:
- Overall churn patterns and rates
- Contract type impact on retention
- Revenue implications of churn
- Customer risk segmentation
- Service adoption correlation with churn